# Single-tool eval: `list_services`

Agent loop built from scratch (no `agent.agent` import), bound to **only** the `list_services` tool from `agent.tools.docker`. Everything else the full agent could reach for is unavailable. This measures how far the model gets on this backend alone.


In [ ]:
from eval_transcript import REFUSAL, run_eval_repeats
from agent.tools.docker import list_services


# Runs per question. One is enough at temperature 0: the loop is
# deterministic, so repeats reproduce the same answer and the same cost.
RUNS = 1


In [ ]:
TOOLS = [list_services]

SYSTEM_PROMPT = """You are Oracle, a terse infrastructure-operations assistant for the Marconi lab.
You have exactly ONE tool available: `list_services` — List known infrastructure targets by their real names: Docker container names and configured monitored endpoints. Call this FIRST when you are unsure of the exact service name, then use the names it returns for your queries.
You are read-only. Never suggest you can restart or change anything yourself;
recommend the action for a human operator instead.
If this one tool can't answer the question, say so plainly instead of guessing.
Be concise."""


### The loop

Every run below goes through `eval_transcript.run_eval_loop`, shared across all the eval notebooks: it sends the messages, dispatches tool calls, prints each call's token counts, and — on every run — dumps the full transcript (every message sent, every response, every tool call/result, per-call token usage) to `eval/logs/<timestamp>__<label>.json`.


## In-scope question

A question `list_services` alone should be able to answer.


In [ ]:
summary = await run_eval_repeats(
    "01_list_services:live-inscope",
    TOOLS,
    SYSTEM_PROMPT,
    "What containers and monitored endpoints exist in this lab?",
    expect_tools=["list_services"],
    runs=RUNS,
)


## Out-of-scope question

A question that needs evidence outside this backend. Watch whether the model admits the limit or hallucinates past it.


In [ ]:
summary = await run_eval_repeats(
    "01_list_services:live-outofscope",
    TOOLS,
    SYSTEM_PROMPT,
    "Is the agent-api container currently healthy?",
    expect=[REFUSAL],
    runs=RUNS,
)
